In [ ]:
!pip install -q transformers accelerate bitsandbytes peft datasets kagglehub scikit-learn pynvml sentencepiece

In [ ]:
import os
import json
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm import tqdm

import torch

from datasets import Dataset

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_curve,
    auc
)

from transformers import (
    InstructBlipProcessor,
    InstructBlipForConditionalGeneration,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)


device = "cuda" if torch.cuda.is_available() else "cpu"

print("DEVICE:", device)

In [ ]:
import kagglehub


dataset_path = kagglehub.dataset_download(
    "parthplc/facebook-hateful-meme-dataset"
)

print(dataset_path)

In [ ]:
TRAIN_FILE = os.path.join(
    dataset_path,
    "data/train.jsonl"
)

DEV_FILE = os.path.join(
    dataset_path,
    "data/dev.jsonl"
)

IMAGE_DIR = os.path.join(
    dataset_path,
    "data"
)

print(TRAIN_FILE)
print(DEV_FILE)

In [ ]:
def load_jsonl(path):

    data = []

    with open(path, "r") as f:

        for line in f:

            data.append(
                json.loads(line)
            )

    return data


train_data = load_jsonl(TRAIN_FILE)

dev_data = load_jsonl(DEV_FILE)

print("TRAIN:", len(train_data))
print("DEV:", len(dev_data))

In [ ]:
train_data = train_data[:1000]

dev_data = dev_data[:500]

print("TRAIN SAMPLES:", len(train_data))
print("DEV SAMPLES:", len(dev_data))

In [ ]:
sample = train_data[0]

image_path = os.path.join(
    IMAGE_DIR,
    sample["img"]
)

image = Image.open(image_path).convert("RGB")

plt.figure(figsize=(6,6))

plt.imshow(image)

plt.axis("off")

plt.title(f'LABEL: {sample["label"]}')

plt.show()

print(sample["text"])

In [ ]:
def convert_dataset(data):

    converted = []

    for sample in data:

        image_path = os.path.join(
            IMAGE_DIR,
            sample["img"]
        )

        answer = (
            "Yes"
            if sample["label"] == 1
            else "No"
        )

        converted.append({

            "image": image_path,

            "question": f'''
            Meme Text:
            {sample["text"]}

            Is this meme hateful?

            Answer Yes or No.
            ''',

            "answer": answer
        })

    return converted


train_converted = convert_dataset(
    train_data
)


dev_converted = convert_dataset(
    dev_data
)

print(train_converted[0])

In [ ]:
train_dataset = Dataset.from_list(
    train_converted
)


dev_dataset = Dataset.from_list(
    dev_converted
)

print(train_dataset)

In [ ]:
MODEL_ID = "Salesforce/instructblip-flan-t5-xl"

bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.float16
)


processor = InstructBlipProcessor.from_pretrained(
    MODEL_ID
)


model = InstructBlipForConditionalGeneration.from_pretrained(

    MODEL_ID,

    quantization_config=bnb_config,

    device_map="auto"
)

print("MODEL LOADED")

In [ ]:
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

model = prepare_model_for_kbit_training(
    model
)

lora_config = LoraConfig(

    r=8,

    lora_alpha=16,

    target_modules=[
        "q",
        "k",
        "v"
    ],

    lora_dropout=0.05,

    bias="none",

    task_type="SEQ_2_SEQ_LM"
)

model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

In [ ]:
def preprocess_function(example):

    image = Image.open(
        example["image"]
    ).convert("RGB")


    full_prompt = (
        example["question"] +
        example["answer"]
    )


    inputs = processor(

        images=image,

        text=full_prompt,

        return_tensors="pt",

        padding=True,

        truncation=True,

        max_length=256
    )


    inputs = {

        k: v.squeeze(0)

        for k, v in inputs.items()
    }


    inputs["labels"] = inputs[
        "input_ids"
    ].clone()


    return inputs


train_dataset = train_dataset.map(
    preprocess_function
)


dev_dataset = dev_dataset.map(
    preprocess_function
)

print("PREPROCESSING COMPLETED")

In [ ]:
training_args = TrainingArguments(

    output_dir="./instructblip_output",

    per_device_train_batch_size=1,

    per_device_eval_batch_size=1,

    gradient_accumulation_steps=1,

    num_train_epochs=10,

    learning_rate=2e-4,

    fp16=True,

    logging_steps=5,

    logging_strategy="steps",

    disable_tqdm=False,

    save_strategy="no",

    report_to="none"
)

In [ ]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=dev_dataset
)

print("TRAINER READY")

In [ ]:
trainer.train()

In [ ]:
def predict_instructblip(image_path, text):

    image = Image.open(
        image_path
    ).convert("RGB")


    prompt = f'''
    Meme Text:
    {text}

    Is this meme hateful?

    Answer Yes or No.
    '''


    inputs = processor(

        images=image,

        text=prompt,

        return_tensors="pt"
    ).to(device)


    with torch.no_grad():

        outputs = model.generate(

            **inputs,

            max_new_tokens=10
        )


    response = processor.batch_decode(

        outputs,

        skip_special_tokens=True
    )[0]


    response = response.lower()


    if "yes" in response:

        return 1

    else:

        return 0

In [ ]:
GPU_AVAILABLE = torch.cuda.is_available()

if GPU_AVAILABLE:

    try:

        from pynvml import *

        nvmlInit()

        handle = nvmlDeviceGetHandleByIndex(0)

        ENERGY_MODE = "GPU"

    except:

        ENERGY_MODE = "CPU"

else:

    ENERGY_MODE = "CPU"



def get_power_usage():

    if ENERGY_MODE == "GPU":

        return (
            nvmlDeviceGetPowerUsage(handle)
            / 1000
        )

    else:

        return 65



def compute_energy_kwh(
    power_watts,
    duration_seconds
):

    return (
        power_watts *
        duration_seconds
    ) / (1000 * 3600)

In [ ]:
model.eval()


y_true = []

y_pred = []

y_scores = []

inference_times = []

energy_consumptions = []


for sample in tqdm(dev_data):

    image_path = os.path.join(
        IMAGE_DIR,
        sample["img"]
    )


    text = sample["text"]

    label = sample["label"]


    start_time = time.time()

    power = get_power_usage()


    prediction = predict_instructblip(
        image_path,
        text
    )


    end_time = time.time()


    duration = end_time - start_time


    energy = compute_energy_kwh(
        power,
        duration
    )


    y_true.append(label)

    y_pred.append(prediction)

    y_scores.append(prediction)

    inference_times.append(duration)

    energy_consumptions.append(energy)

In [ ]:
accuracy = accuracy_score(
    y_true,
    y_pred
)


precision = precision_score(
    y_true,
    y_pred
)


recall = recall_score(
    y_true,
    y_pred
)


f1 = f1_score(
    y_true,
    y_pred
)


avg_time = np.mean(
    inference_times
)


avg_energy = np.mean(
    energy_consumptions
)


print("ACCURACY:", accuracy)

print("PRECISION:", precision)

print("RECALL:", recall)

print("F1-SCORE:", f1)

print("AVERAGE INFERENCE TIME (s):", avg_time)

print("AVERAGE ENERGY (kWh):", avg_energy)

In [ ]:
from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score,

    confusion_matrix,

    classification_report
)


report = classification_report(

    y_true,

    y_pred,

    target_names=[

        "Non-Hateful",

        "Hateful"
    ],

    digits=4
)

print("\nCLASSIFICATION REPORT:\n")

print(report)

In [ ]:
cm = confusion_matrix(
    y_true,
    y_pred
)


plt.figure(figsize=(6,5))

plt.imshow(cm)

plt.colorbar()

plt.title("Confusion Matrix")

plt.xlabel("Predicted")

plt.ylabel("Actual")


for i in range(cm.shape[0]):

    for j in range(cm.shape[1]):

        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )


plt.show()

In [ ]:
fpr, tpr, thresholds = roc_curve(
    y_true,
    y_scores
)


roc_auc = auc(
    fpr,
    tpr
)


plt.figure(figsize=(6,5))

plt.plot(
    fpr,
    tpr,
    label=f"AUC = {roc_auc:.4f}"
)


plt.plot(
    [0,1],
    [0,1],
    linestyle="--"
)


plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve")

plt.legend()

plt.show()

In [ ]:
results_df = pd.DataFrame({

    "Metric": [

        "Accuracy",

        "Precision",

        "Recall",

        "F1-score",

        "ROC-AUC",

        "Inference Time (s)",

        "Energy Consumption (kWh)"
    ],


    "Value": [

        accuracy,

        precision,

        recall,

        f1,

        roc_auc,

        avg_time,

        avg_energy
    ]
})


results_df

In [ ]:
results_df.to_csv(
    "instructblip_results.csv",
    index=False
)

print("RESULTS SAVED")

In [ ]:
model.save_pretrained(
    "instructblip_hateful_memes_lora"
)


processor.save_pretrained(
    "instructblip_hateful_memes_lora"
)

print("MODEL SAVED")